In [1]:
import numpy as np

from ppe_detection_evaluation import FASTER_RCNN, RTDETR_L, SSD300, YOLO26_M, EvaluationFramework

In [2]:
# Ground truth of one image, shared by all models
# (YOLO txt format: normalized cx, cy, w, h / 0-based class id)
image_id = "image_example"
gt_boxes = np.array([
    [0.30, 0.40, 0.10, 0.12],  # Hardhat
    [0.30, 0.60, 0.20, 0.40],  # Person
    [0.70, 0.50, 0.08, 0.10],  # NO-Hardhat
])
gt_labels = np.array([0, 5, 2])

## Faster R-CNN (torchvision)

`out = model([image])[0]` in eval mode:

- `out["boxes"]`: `x1, y1, x2, y2` in pixels of the input image (640×640)
- `out["scores"]`: scores
- `out["labels"]`: class ids where 0 is background, so dataset classes are 1–10 → `label_offset=1`

In [3]:
fw_frcnn = EvaluationFramework(FASTER_RCNN)

pred_boxes = np.array([
    [161.0, 218.0, 224.0, 294.0],
    [127.0, 255.0, 257.0, 513.0],
    [423.0, 289.0, 473.0, 351.0],
])
pred_scores = np.array([0.93, 0.97, 0.81])
pred_labels = np.array([1, 6, 3])

fw_frcnn.add(image_id, pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, latency_ms=85.0)
fw_frcnn.save("results/faster_r-cnn_dummy.json")
fw_frcnn.records[0].pred_labels

array([5, 0, 2])

## YOLO26-M (Ultralytics)

`r = model.predict(image)[0]` on the 640×640 image:

- `r.boxes.xyxy`: `x1, y1, x2, y2` in 640×640 pixels
- `r.boxes.conf`: scores
- `r.boxes.cls`: 0-based class ids (floats)
- `r.speed`: pre-processing / inference / post-processing time in ms

In [4]:
fw_yolo = EvaluationFramework(YOLO26_M)

pred_boxes = np.array([
    [158.0, 219.0, 222.0, 292.0],   # x1, y1, x2, y2
    [130.0, 254.0, 258.0, 509.0],
    [420.0, 290.0, 476.0, 350.0],
    [500.0, 100.0, 560.0, 180.0],   # false positive
])
pred_scores = np.array([0.89, 0.96, 0.78, 0.12])
pred_labels = np.array([0.0, 5.0, 2.0, 8.0])
speed = {"preprocess": 1.5, "inference": 8.0, "postprocess": 1.0}

# add model outputs and other features to the framework
record = fw_yolo.add(image_id, pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, latency_ms=sum(speed.values()))

fw_yolo.save("results/yolo26m_dummy.json")
fw_yolo.records[0].pred_labels

array([5, 0, 2, 8])

## SSD300 (torchvision)

Same output as Faster R-CNN. The model resizes the input to 300×300 internally, but the returned boxes are already mapped back to the 640×640 input image.

In [5]:
fw_ssd = EvaluationFramework(SSD300)

pred_boxes = np.array([
    [160.0, 217.0, 224.0, 294.0],   # x1, y1, x2, y2
    [128.0, 256.0, 256.0, 512.0],
    [533.0, 85.0, 576.0, 139.0],    # false positive; the small NO-Hardhat is missed
])
pred_scores = np.array([0.71, 0.88, 0.35])
pred_labels = np.array([1, 6, 7])   # Hardhat, Person, Safety Cone (+1 for background)

record = fw_ssd.add(image_id, pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, latency_ms=22.0)
fw_ssd.save("results/ssd300_dummy.json")
fw_ssd.records[0].pred_labels

array([5, 0, 6])

## RT-DETR-L (Ultralytics)

Same output as YOLO26-M: `r.boxes.xyxy` in 640×640 pixels, `r.boxes.conf`, and 0-based `r.boxes.cls` (floats).
RT-DETR outputs a fixed number of queries without NMS; predictions below `score_threshold` (0.001) are dropped and at most `max_detections` (100) are kept.

In [6]:
fw_rtdetr = EvaluationFramework(RTDETR_L)

pred_boxes = np.array([
    [159.0, 216.0, 225.0, 295.0],   # x1, y1, x2, y2
    [129.0, 257.0, 255.0, 510.0],
    [421.0, 287.0, 474.0, 353.0],
    [300.0, 300.0, 340.0, 340.0],   # low-score query, dropped
])
pred_scores = np.array([0.91, 0.95, 0.86, 0.0004])
pred_labels = np.array([0.0, 5.0, 2.0, 6.0])

r = fw_rtdetr.add(image_id, pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, latency_ms=12.0)
fw_rtdetr.save("results/rt-detr-l_dummy.json")
fw_rtdetr.records[0].pred_labels

array([5, 0, 2])

## Evaluating

`evaluate()` runs every module in `metrics/` and collects the results in `fw.results`.

| Key | Module | Contents |
|---|---|---|
| `detection` | `compute_detection_ap` | mAP@0.5 / mAP@[.5:.95], per-class AP, AP by object size |
| `violation` | `compute_violation_metrics` | Precision@Recall=0.85, Recall@FPPI=0.1, LAMR and Max F1 for the violation classes |
| `image_level` | `compute_image_level_metrics` | AP / AUROC of image-level violation alerts |
| `confusion` | `compute_ppe_confusion` | PPE present/absent swap rates |
| `latency` | `compute_latency` | Latency (mean / p50 / p95) and FPS |

In [7]:
results = fw_yolo.evaluate()
list(results)

['detection', 'violation', 'image_level', 'confusion', 'latency']

In [8]:
det = results["detection"]
det

{'iou_thresholds': [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95],
 'per_class': {'Hardhat': {'num_gt': 1, 'AP50': 1.0, 'AP75': 1.0, 'AP': 0.8},
  'Mask': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'NO-Hardhat': {'num_gt': 1, 'AP50': 1.0, 'AP75': 1.0, 'AP': 0.8},
  'NO-Mask': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'NO-Safety Vest': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'Person': {'num_gt': 1, 'AP50': 1.0, 'AP75': 1.0, 'AP': 1.0},
  'Safety Cone': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'Safety Vest': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'machinery': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan},
  'vehicle': {'num_gt': 0, 'AP50': nan, 'AP75': nan, 'AP': nan}},
 'groups': {'all': {'mAP50': 1.0, 'mAP': 0.8666666666666667},
  'ppe': {'mAP50': 1.0, 'mAP': 0.8}},
 'by_size': {'small': {'num_gt': {'Hardhat': 0,
    'Mask': 0,
    'NO-Hardhat': 0,
    'NO-Mask': 0,
    'NO-Safety Vest': 0,
    'Person': 0,


In [9]:
vio = results["violation"]
vio

{'settings': {'iou_threshold': 0.5,
  'recall_target': 0.85,
  'fppi_target': 0.1,
  'num_images': 1},
 'per_class': {'NO-Hardhat': {'num_gt': 1,
   'precision_at_recall': 1.0,
   'recall_at_fppi': 1.0,
   'lamr': 9.999999999999996e-11,
   'max_f1': {'f1': 1.0, 'threshold': 0.78, 'precision': 1.0, 'recall': 1.0},
   'max_recall': 1.0},
  'NO-Mask': {'num_gt': 0,
   'precision_at_recall': nan,
   'recall_at_fppi': nan,
   'lamr': nan,
   'max_f1': {'f1': nan, 'threshold': None, 'precision': nan, 'recall': nan},
   'max_recall': 0.0},
  'NO-Safety Vest': {'num_gt': 0,
   'precision_at_recall': nan,
   'recall_at_fppi': nan,
   'lamr': nan,
   'max_f1': {'f1': nan, 'threshold': None, 'precision': nan, 'recall': nan},
   'max_recall': 0.0}},
 'mean': {'precision_at_recall': 1.0,
  'recall_at_fppi': 1.0,
  'lamr': 9.999999999999996e-11,
  'max_recall': 0.3333333333333333,
  'max_f1': 1.0}}

In [10]:
for name, v in results["image_level"].items():
    print(f"{name}: {v}")

any: {'num_images': 1, 'num_positive': 1, 'ap': 1.0, 'auroc': nan, 'precision_at_recall': 1.0, 'max_f1': {'f1': 1.0, 'threshold': 0.78, 'precision': 1.0, 'recall': 1.0}}
NO-Hardhat: {'num_images': 1, 'num_positive': 1, 'ap': 1.0, 'auroc': nan, 'precision_at_recall': 1.0, 'max_f1': {'f1': 1.0, 'threshold': 0.78, 'precision': 1.0, 'recall': 1.0}}
NO-Mask: {'num_images': 1, 'num_positive': 0, 'ap': nan, 'auroc': nan, 'precision_at_recall': nan, 'max_f1': {'f1': nan, 'threshold': None, 'precision': nan, 'recall': nan}}
NO-Safety Vest: {'num_images': 1, 'num_positive': 0, 'ap': nan, 'auroc': nan, 'precision_at_recall': nan, 'max_f1': {'f1': nan, 'threshold': None, 'precision': nan, 'recall': nan}}


In [11]:
for pair, v in results["confusion"].items():
    print(f"{pair}: {v}")

Hardhat -> NO-Hardhat: {'num_gt': 1, 'correct': 1, 'swapped': 0, 'missed': 0, 'swap_rate': 0.0, 'miss_rate': 0.0}
NO-Hardhat -> Hardhat: {'num_gt': 1, 'correct': 1, 'swapped': 0, 'missed': 0, 'swap_rate': 0.0, 'miss_rate': 0.0}
Mask -> NO-Mask: {'num_gt': 0, 'correct': 0, 'swapped': 0, 'missed': 0, 'swap_rate': nan, 'miss_rate': nan}
NO-Mask -> Mask: {'num_gt': 0, 'correct': 0, 'swapped': 0, 'missed': 0, 'swap_rate': nan, 'miss_rate': nan}
Safety Vest -> NO-Safety Vest: {'num_gt': 0, 'correct': 0, 'swapped': 0, 'missed': 0, 'swap_rate': nan, 'miss_rate': nan}
NO-Safety Vest -> Safety Vest: {'num_gt': 0, 'correct': 0, 'swapped': 0, 'missed': 0, 'swap_rate': nan, 'miss_rate': nan}


In [12]:
results["latency"]

{'num_images': 1,
 'mean_ms': 10.5,
 'std_ms': 0.0,
 'p50_ms': 10.5,
 'p95_ms': 10.5,
 'fps': 95.23809523809524}